In [1]:
from eigenstate_solving import BZ_proj
import numpy as np
from band_gap import _sample_for_k
from IPython.display import display
from ipywidgets import FloatSlider, HBox, VBox, interactive_output
from model import square_lattice
from smatrix import create_self_energy_interpolator_numba, t_reg
import matplotlib.pyplot as plt


sigma_data = np.load("../../data/sigma_grid0f1a.npz")
kx = sigma_data["kx"]
ky = sigma_data["ky"]
sigma_grid = sigma_data["sigma_grid"]
sigma_func_period_numba = create_self_energy_interpolator_numba(
    kx, ky, sigma_grid, lattice=square_lattice
)
collective_lamb_shift = sigma_func_period_numba(0, 0).real

Q_para = np.array([0, 0])
E = 2 * (square_lattice.omega_e + collective_lamb_shift)

# labels for W state
p_para = np.array([0, 0])
E1 = (E - 2) / 2

if np.linalg.norm(p_para) + np.linalg.norm(BZ_proj(Q_para - p_para, square_lattice)) > E:
    raise ValueError("In W-state label, the total in-plane photon momenta exceeds the total energy.")

W_eigenvalue = t_reg(p_para, E1, square_lattice, sigma_func_period_numba) * t_reg(
    BZ_proj(Q_para - p_para, square_lattice),
    E - E1,
    square_lattice,
    sigma_func_period_numba,
)

theta = np.linspace(0, 2 * np.pi, 600)


def plot_unit_circle(r_x=0.0, r_y=0.0):
    r_para = np.array([r_x, r_y], dtype=float)

    if np.linalg.norm(r_para) + np.linalg.norm(BZ_proj(Q_para - r_para, square_lattice)) > E:
        raise ValueError("In plane-wave basis, the total in-plane photon momenta exceeds the total energy.")
        return

    tt_image = _sample_for_k(
        r_para,
        Q_para,
        E,
        1e-10,
        100,
        sigma_func_period_numba,
        square_lattice,
    )

    values = np.asarray(tt_image, dtype=complex)
    finite_nonzero = np.isfinite(values.real) & np.isfinite(values.imag) & (np.abs(values) > 0)
    unit_values = values[finite_nonzero]

    fig, ax = plt.subplots(figsize=(6, 6))
    ax.plot(np.cos(theta), np.sin(theta), color="0.75", lw=1.5, label="unit circle")
    ax.scatter(
        unit_values.real,
        unit_values.imag,
        color="tab:blue",
        s=16,
        edgecolor="black",
        linewidth=0.0,
        zorder=3,
    )

    ax.scatter(
        W_eigenvalue.real,
        W_eigenvalue.imag,
        color="red",
        s=16,
        edgecolor="black",
        linewidth=0.0,
        zorder=4,
        label="W_eigenvalue",
    )

    ax.axhline(0, color="0.85", lw=0.8)
    ax.axvline(0, color="0.85", lw=0.8)
    ax.set_aspect("equal", adjustable="box")
    ax.set_xlim(-1.1, 1.1)
    ax.set_ylim(-1.1, 1.1)
    ax.set_xlabel("Re")
    ax.set_ylabel("Im")
    ax.set_title(f"tt_image phases on the unit circle, r_para=({r_x:.1f}, {r_y:.1f})")
    ax.legend(loc="best")
    plt.show()


r_x_slider = FloatSlider(
    value=0.0,
    min=-100.0,
    max=100.0,
    step=1.0,
    description="r_x",
    continuous_update=False,
)
r_y_slider = FloatSlider(
    value=0.0,
    min=-100.0,
    max=100.0,
    step=1.0,
    description="r_y",
    continuous_update=False,
)

plot_output = interactive_output(plot_unit_circle, {"r_x": r_x_slider, "r_y": r_y_slider})
display(VBox([HBox([r_x_slider, r_y_slider]), plot_output]))
